# Baseline U-Net34 — ISIC 2018 Task 1 trên Kaggle (2 × T4)

Notebook này **tự chứa toàn bộ pipeline** và chỉ đọc trực tiếp dataset đã Attach tại ` /kaggle/input/skin-lesion-segmentation-hus `. Không dùng `kagglehub`, không dùng `./data`, không import code từ `src`.

Giao thức chống data leakage:

1. Chỉ tập **Training chính thức** được chia ngẫu nhiên, cố định seed thành 3 fold.
2. Với mỗi fold: 2 fold là train (có augmentation), fold còn lại là holdout sạch để chọn checkpoint **Best Dice**.
3. LR Finder chỉ nhận train subset; không nhận holdout, validation chính thức hoặc test.
4. Validation chính thức chỉ được báo cáo sau khi cả 3 model đã được chọn. Test chỉ chạy ở cuối.
5. Ensemble là trung bình xác suất với trọng số bằng nhau; không fit trọng số trên validation/test.

> Trên Kaggle: bấm **Add Input**, thêm dataset `tinlngnguyn/skin-lesion-segmentation-hus`, rồi chọn Accelerator **GPU T4 x2**. Encoder dùng trọng số ImageNet nên cần bật Internet cho lần tải đầu tiên, hoặc trọng số phải có sẵn trong cache. Notebook không âm thầm đổi sang random initialization nếu tải pretrained thất bại.

In [ ]:
import copy
import hashlib
import math
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm

# -------------------- Cấu hình baseline --------------------
DATA_ROOT = Path('/kaggle/input/skin-lesion-segmentation-hus')
OUTPUT_ROOT = Path('/kaggle/working')
BEST_CHECKPOINT_PATH = OUTPUT_ROOT / 'unet34_best_checkpoint.pt'
SEED = 42
N_FOLDS = 3
SIZES = (128, 256)
FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 30
GLOBAL_BATCH_SIZE = 8       # batch toàn cục; DataParallel chia 4 mẫu/T4
NUM_WORKERS = min(4, os.cpu_count() or 2)
LRF_ITERS = 100
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
CUT_FRAC, STLR_RATIO = 0.1, 32.0
WEIGHT_DECAY = 0.0
PRED_THRESHOLD = 0.5
JACCARD_CUT = 0.65
PRETRAINED = True
REQUIRE_T4_X2 = True
FAST_DEV_RUN = False        # True chỉ để kiểm tra nhanh code; mặc định là train đầy đủ

if FAST_DEV_RUN:
    SIZES = (128,)
    FROZEN_EPOCHS = UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed()
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Kaggle Accelerator = GPU T4 x2 trước khi train.')
DEVICE = torch.device('cuda:0')
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
if REQUIRE_T4_X2 and (GPU_COUNT != 2 or not all('T4' in name.upper() for name in GPU_NAMES)):
    raise RuntimeError(f'Cần đúng GPU T4 x2, nhưng đang thấy {GPU_COUNT} GPU: {GPU_NAMES}')
USE_AMP = False  # stability first: avoid FP16 ConvTranspose2d illegal access on T4 x2
PIN_MEMORY = True
print({'device': str(DEVICE), 'gpus': GPU_NAMES, 'global_batch': GLOBAL_BATCH_SIZE,
       'batch_per_gpu': GLOBAL_BATCH_SIZE // GPU_COUNT, 'workers': NUM_WORKERS,
       'amp': USE_AMP, 'best_checkpoint': str(BEST_CHECKPOINT_PATH)})


## 1. Khám phá và kiểm tra dữ liệu

Dataset Kaggle này có hai lớp thư mục trùng tên. Hàm dưới đây chỉ nhận thư mục thực sự chứa file ảnh, sau đó ghép cặp nghiêm ngặt `ID.jpg ↔ ID_segmentation.png`. Ảnh chỉ được mở trong `Dataset.__getitem__`, vì vậy không nạp 13.9 GB vào RAM.

In [ ]:
SPLIT_DIRECTORIES = {
    'train': ('ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth'),
    'validation': ('ISIC2018_Task1-2_Validation_Input', 'ISIC2018_Task1_Validation_GroundTruth'),
    'test': ('ISIC2018_Task1-2_Test_Input', 'ISIC2018_Task1_Test_GroundTruth'),
}
EXPECTED_COUNTS = {'train': 2594, 'validation': 100, 'test': 1000}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_populated_directory(name, suffixes):
    outer = DATA_ROOT / name
    candidates = [outer / name, outer]
    if DATA_ROOT.is_dir():
        candidates.extend(path for path in DATA_ROOT.rglob(name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        if any(path.is_file() and path.suffix.lower() in suffixes for path in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(f'Không tìm thấy thư mục có dữ liệu: {name} dưới {DATA_ROOT}')

def discover_records(split):
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {'.png'})
    image_map = {p.stem: p for p in image_dir.iterdir()
                 if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES}
    mask_map = {p.stem.removesuffix('_segmentation'): p for p in mask_dir.iterdir()
                if p.is_file() and p.suffix.lower() == '.png' and p.stem.endswith('_segmentation')}
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(f'{split}: missing_masks={missing_masks[:5]}, orphan_masks={orphan_masks[:5]}')
    records = [
        {'image_id': image_id, 'image_path': image_map[image_id], 'mask_path': mask_map[image_id]}
        for image_id in sorted(image_map)
    ]
    if len(records) != EXPECTED_COUNTS[split]:
        raise RuntimeError(f'{split}: cần {EXPECTED_COUNTS[split]} cặp, tìm thấy {len(records)}')
    return records, image_dir, mask_dir

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Chưa Attach dataset hoặc sai slug: {DATA_ROOT}')
train_records_all, train_image_dir, train_mask_dir = discover_records('train')
official_val_records_all, val_image_dir, val_mask_dir = discover_records('validation')
test_records_all, test_image_dir, test_mask_dir = discover_records('test')

split_ids = {
    'train': {r['image_id'] for r in train_records_all},
    'validation': {r['image_id'] for r in official_val_records_all},
    'test': {r['image_id'] for r in test_records_all},
}
assert split_ids['train'].isdisjoint(split_ids['validation'])
assert split_ids['train'].isdisjoint(split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])

def deterministic_subset(records, count, seed):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))[:min(count, len(records))]
    return [records[int(i)] for i in indices]

if FAST_DEV_RUN:
    train_records = deterministic_subset(train_records_all, 96, SEED)
    official_val_records = deterministic_subset(official_val_records_all, 24, SEED + 1)
    test_records = deterministic_subset(test_records_all, 24, SEED + 2)
else:
    train_records = train_records_all
    official_val_records = official_val_records_all
    test_records = test_records_all

def make_folds(records, n_splits=N_FOLDS, seed=SEED):
    permutation = torch.randperm(
        len(records), generator=torch.Generator().manual_seed(int(seed))
    ).tolist()
    fold_sizes = [len(records) // n_splits] * n_splits
    for index in range(len(records) % n_splits):
        fold_sizes[index] += 1
    folds = []
    all_indices = set(range(len(records)))
    offset = 0
    for fold_size in fold_sizes:
        holdout_indices = set(permutation[offset:offset + fold_size])
        train_indices = all_indices - holdout_indices
        assert train_indices.isdisjoint(holdout_indices)
        assert train_indices | holdout_indices == all_indices
        folds.append(([records[i] for i in sorted(train_indices)],
                      [records[i] for i in sorted(holdout_indices)]))
        offset += fold_size
    return folds

folds = make_folds(train_records)
print('counts:', len(train_records), len(official_val_records), len(test_records))
print('folds:', [(len(tr), len(ho)) for tr, ho in folds])
print('train dirs:', train_image_dir, train_mask_dir, sep='\n  ')
print('validation dirs:', val_image_dir, val_mask_dir, sep='\n  ')
print('test dirs:', test_image_dir, test_mask_dir, sep='\n  ')


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def gray_world(image, **kwargs):
    del kwargs
    image_float = image.astype(np.float32)
    channel_means = image_float.mean(axis=(0, 1))
    gray_mean = channel_means.mean()
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    corrected = image_float * scale.reshape(1, 1, 3)
    return np.clip(corrected, 0, 255).astype(np.uint8)

def build_transform(size, train, seed):
    steps = [
        A.Resize(size, size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.Lambda(image=gray_world, p=1.0),
    ]
    if train:
        steps.extend([
            A.D4(p=1.0),
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05), interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
        ])
    steps.extend([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
    return A.Compose(steps, seed=int(seed))

class ISICSegmentationDataset(Dataset):
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        self.transform = build_transform(self.size, self.train, seed=seed)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(record['image_path']) as source:
            image_np = np.asarray(source.convert('RGB'), dtype=np.uint8).copy()
        with Image.open(record['mask_path']) as source:
            mask_np = np.asarray(source.convert('L'), dtype=np.uint8).copy()
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Kích thước ảnh/mask lệch cho {record['image_id']}: "
                             f'{image_np.shape[:2]} và {mask_np.shape}')

        transformed = self.transform(image=image_np, mask=(mask_np > 127).astype(np.uint8))
        image, mask = transformed['image'].float(), transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        mask = (mask > 0).float()
        return image.contiguous(), mask.contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, seed, drop_last=None, persistent=None):
    if drop_last is None:
        drop_last = False
    if persistent is None:
        persistent = NUM_WORKERS > 0
    generator = torch.Generator().manual_seed(int(seed))
    return DataLoader(
        ISICSegmentationDataset(records, size=size, train=train, seed=seed),
        batch_size=GLOBAL_BATCH_SIZE, shuffle=train, drop_last=bool(drop_last),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=bool(persistent and NUM_WORKERS > 0),
        worker_init_fn=seed_worker, generator=generator,
    )

# Self-check: clean transform phải deterministic; mask chỉ có 0/1 và không rỗng/toàn ảnh.
clean_ds = ISICSegmentationDataset(train_records[:3], size=SIZES[0], train=False)
for i in range(len(clean_ds)):
    x1, y1 = clean_ds[i]
    x2, y2 = clean_ds[i]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.shape == (3, SIZES[0], SIZES[0]) and y1.shape == (1, SIZES[0], SIZES[0])
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_ds
print('Dataset/transform checks: OK')


In [ ]:
preview_ds = ISICSegmentationDataset(train_records[:3], size=256, train=False)
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(3, 2, figsize=(8, 11))
for row in range(3):
    image, mask = preview_ds[row]
    display_image = (image * std + mean).clamp(0, 1).permute(1, 2, 0)
    axes[row, 0].imshow(display_image)
    axes[row, 0].set_title(train_records[row]['image_id'])
    axes[row, 1].imshow(mask[0], cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'mask, foreground={mask.mean():.3f}')
    axes[row, 0].axis('off'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()
del preview_ds


## 2. U-Net34

Encoder là ResNet34 pretrained ImageNet và decoder khớp baseline trong dự án. Stage `frozen` chỉ đóng băng non-BN của `stem + layer1 + layer2`; `layer3`, `layer4` và decoder vẫn học. Trước mỗi stage frozen, trạng thái `requires_grad` được reset để không mang nhầm BN bị khóa từ độ phân giải trước. Stage `unfrozen` mở toàn mạng nhưng cố định toàn bộ BatchNorm (cả affine parameters lẫn running statistics).

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch // 2, 2, stride=2)
        self.skip_projection = nn.Conv2d(skip_ch, out_ch // 2, 1)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        skip = self.skip_projection(skip)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        return self.bn(F.relu(torch.cat([x, skip], dim=1)))

class UNet34(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = models.resnet34(weights=weights)
        except Exception as exc:
            raise RuntimeError(
                'Không tải được ResNet34 ImageNet weights. Bật Internet trong Kaggle Settings '
                'và chạy lại cell; notebook không tự đổi sang random initialization.'
            ) from exc
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)
        self.head = nn.Conv2d(16, 1, 1)

    @property
    def first_layer_group(self):
        # Avoid registering aliases that duplicate encoder tensors in the checkpoint.
        return (self.stem, self.layer1, self.layer2)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bottleneck = self.layer4(s3)
        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)
        return self.head(decoded)

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def to_runtime(model):
    model = model.to(DEVICE)
    return nn.DataParallel(model, device_ids=list(range(GPU_COUNT))) if GPU_COUNT > 1 else model

def freeze_first_layer_group(model):
    base = unwrap_model(model)
    # Reset trạng thái từ stage trước, rồi mới freeze nhóm đầu. Điều này đặc biệt
    # quan trọng khi chuyển 128 -> 256 vì stage unfrozen trước đó đã khóa mọi BN.
    for parameter in base.parameters():
        parameter.requires_grad = True
    for module in base.first_layer_group:
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)

def unfreeze_with_frozen_bn(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False

def set_train_mode(model, bn_frozen):
    model.train()
    if bn_frozen:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

def cpu_state_dict(model):
    return {key: value.detach().cpu().clone() for key, value in unwrap_model(model).state_dict().items()}

def load_model_state(model, state):
    while state and all(key.startswith('module.') for key in state):
        state = {key.removeprefix('module.'): value for key, value in state.items()}
    unwrap_model(model).load_state_dict(state)

shape_model = UNet34(pretrained=False).eval()
with torch.inference_mode():
    shape_output = shape_model(torch.randn(2, 3, 128, 128))
assert shape_output.shape == (2, 1, 128, 128)
print(f'params={sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M, output={tuple(shape_output.shape)}')
del shape_model, shape_output


In [ ]:
criterion = nn.BCEWithLogitsLoss()

def per_image_scores(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    pred_sum = prediction.sum(dims)
    target_sum = target.sum(dims)
    union = pred_sum + target_sum - intersection
    jaccard = torch.where(union > 0, intersection / union.clamp(min=1), torch.ones_like(union))
    dice = torch.where(pred_sum + target_sum > 0,
                       2 * intersection / (pred_sum + target_sum).clamp(min=1),
                       torch.ones_like(union))
    sensitivity = torch.where(target_sum > 0, intersection / target_sum.clamp(min=1),
                              torch.ones_like(target_sum))
    return jaccard, dice, sensitivity

def surface(mask):
    eroded = ndimage.binary_erosion(mask, border_value=0)
    return np.logical_xor(mask, eroded)

def hd95_single(prediction, target):
    prediction, target = prediction.astype(bool), target.astype(bool)
    height, width = target.shape
    if not prediction.any() and not target.any():
        return 0.0
    if not prediction.any() or not target.any():
        return float(math.hypot(height, width))
    pred_surface, target_surface = surface(prediction), surface(target)
    distance_to_target = ndimage.distance_transform_edt(~target_surface)
    distance_to_pred = ndimage.distance_transform_edt(~pred_surface)
    distances = np.concatenate([distance_to_target[pred_surface], distance_to_pred[target_surface]])
    return float(np.percentile(distances, 95)) if distances.size else 0.0

@torch.inference_mode()
def ensemble_probability(models_list, inputs):
    total = torch.zeros((inputs.shape[0], 1, inputs.shape[2], inputs.shape[3]),
                        device=inputs.device, dtype=torch.float32)
    for model in models_list:
        model.eval()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            total += torch.sigmoid(model(inputs).float())
    return total / len(models_list)

@torch.inference_mode()
def evaluate(models_list, loader, include_hd95=False):
    sums = {'jaccard': 0.0, 'threshold_jaccard': 0.0, 'dice': 0.0, 'sensitivity': 0.0}
    hd95_sum, sample_count = 0.0, 0
    for inputs, targets in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets_device = targets.to(DEVICE, non_blocking=True)
        probability = ensemble_probability(models_list, inputs)
        jaccard, dice, sensitivity = per_image_scores(probability, targets_device)
        sums['jaccard'] += float(jaccard.sum())
        sums['threshold_jaccard'] += float((jaccard * (jaccard >= JACCARD_CUT)).sum())
        sums['dice'] += float(dice.sum())
        sums['sensitivity'] += float(sensitivity.sum())
        sample_count += targets.shape[0]
        if include_hd95:
            pred_np = (probability > PRED_THRESHOLD).cpu().numpy()[:, 0]
            target_np = (targets.numpy()[:, 0] > 0.5)
            hd95_sum += sum(hd95_single(pred, target) for pred, target in zip(pred_np, target_np))
    metrics = {key: value / sample_count for key, value in sums.items()}
    if include_hd95:
        metrics['hd95'] = hd95_sum / sample_count
    return metrics

# Metric self-check: IoU [1.0, 0.5] -> threshold-Jaccard (1 + 0) / 2 = 0.5.
metric_target = torch.zeros(2, 1, 8, 8)
metric_target[0, :, :4] = 1; metric_target[1, :, :, :4] = 1
metric_prob = metric_target.clone(); metric_prob[1] = 0; metric_prob[1, :, :, :2] = 1
metric_jac, metric_dice, _ = per_image_scores(metric_prob, metric_target)
assert torch.allclose(metric_jac, torch.tensor([1.0, 0.5]))
assert abs(float((metric_jac * (metric_jac >= JACCARD_CUT)).mean()) - 0.5) < 1e-6
del metric_target, metric_prob, metric_jac, metric_dice
print('Metric checks: OK')


## 3. LR Finder, STLR và Best Dice

Mỗi độ phân giải chạy hai stage: `frozen` rồi `unfrozen`. LR Finder có loader train riêng; sau khi dò, model (kể cả BatchNorm buffers) được khôi phục và loader train được tạo lại. Holdout sạch chỉ dùng để chọn epoch Best Dice. Best của stage trước được tính như epoch 0 của stage sau, nên unfreeze không thể vô tình ghi đè một checkpoint frozen tốt hơn.

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def trainable_parameters(model):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise RuntimeError('Không có parameter nào đang trainable')
    return parameters

def optimizer_for(model, learning_rate):
    return torch.optim.Adam(trainable_parameters(model), lr=learning_rate, weight_decay=WEIGHT_DECAY)

def train_batch(model, optimizer, scaler, inputs, targets):
    inputs = inputs.to(DEVICE, non_blocking=True)
    targets = targets.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        loss = criterion(model(inputs).float(), targets)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach())

def lr_find(model, train_records_fold, size, stage_seed, bn_frozen):
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    optimizer = optimizer_for(model, LR_MIN)
    scaler = make_scaler()
    loader = make_loader(train_records_fold, size=size, train=True, seed=stage_seed, drop_last=False)
    multiplier = (LR_MAX / LR_MIN) ** (1 / max(1, LRF_ITERS - 1))
    iterator = iter(loader)
    learning_rates, losses = [], []
    moving_average, best_loss = 0.0, float('inf')
    for iteration in range(LRF_ITERS):
        try:
            inputs, targets = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            inputs, targets = next(iterator)
        learning_rate = LR_MIN * multiplier ** iteration
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        set_train_mode(model, bn_frozen=bn_frozen)
        raw_loss = train_batch(model, optimizer, scaler, inputs, targets)
        moving_average = LRF_BETA * moving_average + (1 - LRF_BETA) * raw_loss
        smoothed_loss = moving_average / (1 - LRF_BETA ** (iteration + 1))
        if not np.isfinite(smoothed_loss) or (iteration > 5 and smoothed_loss > 4 * best_loss):
            break
        best_loss = min(best_loss, smoothed_loss)
        learning_rates.append(learning_rate)
        losses.append(smoothed_loss)
    del iterator, loader, optimizer, scaler
    load_model_state(model, original_state)
    set_seed(stage_seed)
    learning_rates = np.asarray(learning_rates, dtype=float)
    losses = np.asarray(losses, dtype=float)
    if len(learning_rates) < 3:
        raise RuntimeError('LR Finder dừng quá sớm, không đủ điểm để chọn learning rate')
    gradient = np.gradient(losses, np.log10(learning_rates))
    selected_lr = float(learning_rates[int(np.argmin(gradient))])
    return selected_lr, learning_rates.tolist(), losses.tolist()

def stlr_lambda(total_steps, cut_frac=CUT_FRAC, ratio=STLR_RATIO):
    cut = max(1, int(total_steps * cut_frac))
    def schedule(step):
        if step < cut:
            progress = step / cut
        else:
            denominator = cut * (1 / cut_frac - 1)
            progress = max(0.0, 1 - (step - cut) / max(1, denominator))
        return (1 + progress * (ratio - 1)) / ratio
    return schedule

def run_stage(model, train_records_fold, holdout_records, size, epochs, bn_frozen,
              fold_number, stage_name, stage_seed):
    selected_lr, lrf_lrs, lrf_losses = lr_find(
        model, train_records_fold, size=size, stage_seed=stage_seed, bn_frozen=bn_frozen
    )
    holdout_loader = make_loader(holdout_records, size=size, train=False, seed=stage_seed)
    initial_metrics = evaluate([model], holdout_loader, include_hd95=False)
    best_score = float(initial_metrics['dice'])
    best_metrics, best_epoch = initial_metrics, 0
    best_state = cpu_state_dict(model)
    history = []

    steps_per_epoch = math.ceil(len(train_records_fold) / GLOBAL_BATCH_SIZE)
    if steps_per_epoch < 1:
        raise RuntimeError('Train subset nhỏ hơn batch size')
    optimizer = optimizer_for(model, selected_lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )

    for epoch in range(epochs):
        epoch_loader = make_loader(
            train_records_fold, size=size, train=True, seed=stage_seed + epoch,
            drop_last=False, persistent=False
        )
        set_train_mode(model, bn_frozen=bn_frozen)
        epoch_loss = 0.0
        for inputs, targets in epoch_loader:
            epoch_loss += train_batch(model, optimizer, scaler, inputs, targets)
            scheduler.step()
        epoch_loss /= len(epoch_loader)
        del epoch_loader
        metrics = evaluate([model], holdout_loader, include_hd95=False)
        row = {'fold': fold_number, 'size': size, 'stage': stage_name, 'epoch': epoch + 1,
               'train_loss': epoch_loss, 'lr': optimizer.param_groups[0]['lr'], **metrics}
        history.append(row)
        if float(metrics['dice']) > best_score:
            best_score = float(metrics['dice'])
            best_metrics, best_epoch = metrics, epoch + 1
            best_state = cpu_state_dict(model)
        print(f"[fold {fold_number} | {size}px | {stage_name}] "
              f"epoch {epoch + 1:02d}/{epochs} loss={epoch_loss:.4f} "
              f"dice={metrics['dice']:.4f} jac={metrics['jaccard']:.4f} "
              f"thr-jac={metrics['threshold_jaccard']:.4f}")

    load_model_state(model, best_state)
    result = {
        'fold': fold_number, 'size': size, 'stage': stage_name, 'selected_lr': selected_lr,
        'best_epoch': best_epoch, 'best_metrics': best_metrics, 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
    }
    del best_state
    return result


In [ ]:
# Preflight một batch thật: kiểm tra I/O, augmentation, loss, backward FP32 và DataParallel.
set_seed(SEED)
probe_model = to_runtime(UNet34(pretrained=False))
assert isinstance(probe_model, nn.DataParallel) and probe_model.device_ids == [0, 1]
freeze_first_layer_group(probe_model)
probe_optimizer = optimizer_for(probe_model, 1e-4)
probe_scaler = make_scaler()
probe_loader = make_loader(train_records[:max(16, GLOBAL_BATCH_SIZE)], size=SIZES[0],
                           train=True, seed=SEED, drop_last=False, persistent=False)
probe_inputs, probe_targets = next(iter(probe_loader))
probe_loss = train_batch(probe_model, probe_optimizer, probe_scaler, probe_inputs, probe_targets)
torch.cuda.synchronize()  # surface any asynchronous CUDA error in this cell
assert np.isfinite(probe_loss)
print(f'2-GPU preflight OK: batch={tuple(probe_inputs.shape)}, loss={probe_loss:.4f}')
del probe_model, probe_optimizer, probe_scaler, probe_loader, probe_inputs, probe_targets
torch.cuda.empty_cache()
set_seed(SEED)


## 4. Train 3 fold: 128 → 256, frozen → unfrozen

Cell này là phần tốn thời gian. Mỗi fold chạy 120 epoch, tổng cộng 360 epoch. Notebook chỉ ghi một file `/kaggle/working/unet34_best_checkpoint.pt`; `/kaggle/input` luôn chỉ đọc.

In [ ]:
RUN_CONFIG = {
    'seed': SEED, 'n_folds': N_FOLDS, 'sizes': list(SIZES),
    'frozen_epochs': FROZEN_EPOCHS, 'unfrozen_epochs': UNFROZEN_EPOCHS,
    'global_batch_size': GLOBAL_BATCH_SIZE, 'train_drop_last': False, 'pretrained': PRETRAINED,
    'loss': 'BCEWithLogitsLoss', 'optimizer': 'Adam', 'weight_decay': WEIGHT_DECAY,
    'selection_metric': 'dice', 'fast_dev_run': FAST_DEV_RUN,
    'lr_finder': {'min': LR_MIN, 'max': LR_MAX, 'iterations': LRF_ITERS, 'beta': LRF_BETA},
    'stlr': {'cut_frac': CUT_FRAC, 'ratio': STLR_RATIO},
    'augmentation': 'albumentations2_resize+grayworld+d4+affine44_zoom1.05+rbc0.2',
    'model': 'project_unet34_resnet34_imagenet_v1',
    'prediction_threshold': PRED_THRESHOLD, 'jaccard_cut': JACCARD_CUT,
    'amp': USE_AMP, 'splitter': 'torch_randperm_v1', 'protocol_version': 4,
}
def ids_digest(records):
    joined = '\n'.join(record['image_id'] for record in records)
    return hashlib.sha256(joined.encode('utf-8')).hexdigest()

def train_fold(fold_number, train_fold_records, holdout_records):
    fold_seed = SEED + fold_number - 1
    set_seed(fold_seed)
    model = to_runtime(UNet34(pretrained=PRETRAINED))
    stage_results = []
    for size_index, size in enumerate(SIZES):
        freeze_first_layer_group(model)
        frozen_result = run_stage(
            model, train_fold_records, holdout_records, size=size, epochs=FROZEN_EPOCHS,
            bn_frozen=False, fold_number=fold_number, stage_name='frozen',
            stage_seed=fold_seed + size_index * 1000,
        )
        stage_results.append(frozen_result)
        unfreeze_with_frozen_bn(model)
        unfrozen_result = run_stage(
            model, train_fold_records, holdout_records, size=size, epochs=UNFROZEN_EPOCHS,
            bn_frozen=True, fold_number=fold_number, stage_name='unfrozen',
            stage_seed=fold_seed + size_index * 1000 + 500,
        )
        stage_results.append(unfrozen_result)

    final_loader = make_loader(holdout_records, size=SIZES[-1], train=False, seed=fold_seed)
    holdout_metrics = evaluate([model], final_loader, include_hd95=False)
    meta = {
        'fold': fold_number, 'fold_seed': fold_seed, 'config': RUN_CONFIG,
        'train_count': len(train_fold_records), 'holdout_count': len(holdout_records),
        'train_ids_sha256': ids_digest(train_fold_records),
        'holdout_ids_sha256': ids_digest(holdout_records),
        'holdout_metrics': holdout_metrics, 'stage_results': stage_results,
    }
    base_model = unwrap_model(model).cpu()
    del model, final_loader
    torch.cuda.empty_cache()
    return base_model, meta

fold_models_cpu, fold_meta = [], []
training_started = time.time()
for fold_number, (train_fold_records, holdout_records) in enumerate(folds, start=1):
    print(f'\n===== FOLD {fold_number}/{N_FOLDS}: train={len(train_fold_records)}, holdout={len(holdout_records)} =====')
    model_cpu, meta = train_fold(fold_number, train_fold_records, holdout_records)
    fold_models_cpu.append(model_cpu)
    fold_meta.append(meta)

elapsed_hours = (time.time() - training_started) / 3600
fold_table = pd.DataFrame([{'fold': meta['fold'], **meta['holdout_metrics']} for meta in fold_meta])
selected_single_index = int(np.argmax([meta['holdout_metrics']['dice'] for meta in fold_meta]))
selected_single_fold = selected_single_index + 1
best_meta = {
    **fold_meta[selected_single_index],
    'selection': 'highest clean fold-holdout Dice among 3 folds',
    'all_fold_holdout_metrics': fold_table.to_dict(orient='records'),
}
atomic_torch_save({
    'model_state': cpu_state_dict(fold_models_cpu[selected_single_index]),
    'meta': best_meta,
}, BEST_CHECKPOINT_PATH)
display(fold_table)
print(f'Thời gian train: {elapsed_hours:.2f} giờ')
print(f'Best checkpoint: {BEST_CHECKPOINT_PATH} (fold {selected_single_fold})')


## 5. Đánh giá official validation, chọn single BestDice và test ensemble

Mỗi `fold model` đã là checkpoint Best Dice trên holdout tương ứng. Nếu cần một model đơn, notebook chọn fold có holdout Dice cao nhất — **không nhìn official validation/test**. Official validation được dùng để báo cáo từng model và ensemble; test báo cáo model đơn đã khóa và ensemble 3 fold.

In [ ]:
runtime_models = [to_runtime(model) for model in fold_models_cpu]
validation_loader = make_loader(official_val_records, size=SIZES[-1], train=False, seed=SEED)
test_loader = make_loader(test_records, size=SIZES[-1], train=False, seed=SEED)

validation_rows = []
for index, model in enumerate(runtime_models, start=1):
    metrics = evaluate([model], validation_loader, include_hd95=True)
    validation_rows.append({'model': f'BestDice fold {index}', **metrics})
validation_ensemble = evaluate(runtime_models, validation_loader, include_hd95=True)
validation_rows.append({'model': 'Equal-weight ensemble', **validation_ensemble})
validation_table = pd.DataFrame(validation_rows)
selected_single_index = int(np.argmax([meta['holdout_metrics']['dice'] for meta in fold_meta]))
selected_single_fold = selected_single_index + 1
test_single = evaluate([runtime_models[selected_single_index]], test_loader, include_hd95=True)
test_ensemble = evaluate(runtime_models, test_loader, include_hd95=True)
test_table = pd.DataFrame([
    {'model': f'Single BestDice fold {selected_single_fold}', **test_single},
    {'model': 'Equal-weight ensemble', **test_ensemble},
])
display(validation_table.style.format({column: '{:.4f}' for column in validation_table.columns if column != 'model'}))
display(test_table.style.format({column: '{:.4f}' for column in test_table.columns if column != 'model'}))
print('Single model được khóa từ CV: fold', selected_single_fold)


In [ ]:
# Learning curves và LR Finder của fold 1.
stage_results = fold_meta[0]['stage_results']
fig, axes = plt.subplots(len(stage_results), 2, figsize=(12, 3.2 * len(stage_results)))
for row, result in enumerate(stage_results):
    axes[row, 0].plot(result['lrf_lrs'], result['lrf_losses'])
    axes[row, 0].set_xscale('log')
    axes[row, 0].axvline(result['selected_lr'], color='red', linestyle='--')
    axes[row, 0].set_title(f"{result['size']}px {result['stage']} — LR Finder")
    axes[row, 0].set_xlabel('learning rate'); axes[row, 0].set_ylabel('smoothed train loss')
    history = pd.DataFrame(result['history'])
    if not history.empty:
        axes[row, 1].plot(history['epoch'], history['dice'], label='Dice')
        axes[row, 1].plot(history['epoch'], history['jaccard'], label='Jaccard')
        axes[row, 1].axvline(result['best_epoch'], color='red', linestyle='--', label='Best Dice')
    axes[row, 1].set_title(f"{result['size']}px {result['stage']} — clean holdout")
    axes[row, 1].set_xlabel('epoch'); axes[row, 1].legend()
plt.tight_layout()
plt.show()
print('Saved artifact:', BEST_CHECKPOINT_PATH)


## Cách đọc kết quả

- Pretrained ResNet34 chỉ khởi tạo **encoder**; decoder/head vẫn ngẫu nhiên. Vì vậy Dice bằng 0 ở những epoch đầu không chứng minh pretrained bị lỗi.
- Ảnh được đổi sang `[0, 1]`, cân bằng màu Gray World, sau đó chuẩn hóa ImageNet. Tensor đưa vào model vì thế không còn bị giới hạn trong `[0, 1]`; đây là đúng với pretrained encoder.
- Dice bằng 1.0 liên tục ngay từ đầu thường đáng kiểm tra. Các cell audit đã kiểm tra pairing, split không giao nhau, mask nhị phân/không rỗng và clean transform deterministic.
- `num_workers` chỉ điều khiển số tiến trình CPU đọc/biến đổi ảnh; việc dùng 2 GPU do `nn.DataParallel` quyết định. `GLOBAL_BATCH_SIZE=8` là tổng batch, không phải batch cho mỗi GPU.
- Nếu OOM, giảm `GLOBAL_BATCH_SIZE` xuống 4. Không tăng batch chỉ vì có hai GPU nếu muốn giữ đúng baseline.